# SFT training + eval on Colab — one-shot run

**Requires GPU runtime.** Runtime → Change runtime type → T4 GPU.

1. Setup (clone + install)
2. Train SFT on 90 pairs (1 small GS-producing blueprint, 1-3 edits each) — ~10 min on T4
3. Eval policy_0 (base) vs policy_1 (new SFT) — ~30 min
4. Print result table

**Run all cells.** Come back in ~45 min.


In [ ]:
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project 2>/dev/null || (cd /content/ml_project && git pull)
%cd /content/ml_project
!pip install -q -r requirements-colab.txt
# Remove Colab broken packages AFTER install so they stay gone even if pip pulled them back.
# bitsandbytes: mixed-version native lib that segfaults on import; not needed (fp16).
# torchvision: torch/torchvision ABI mismatch; not needed (text-only project).
!pip uninstall -y -q bitsandbytes torchvision

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Train SFT (fresh, replaces existing ckpts/sft)


In [ ]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True rm -rf ckpts/sft && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m training.train_sft --output-dir ./ckpts/sft --num-train-epochs 3


## Eval policy_0 vs policy_1 (fresh SFT)


In [ ]:
!python -m training.evaluate \
  --checkpoints policy_0=BASE policy_1=./ckpts/sft \
  --samples-per-layout 4 --n-val 20 \
  --out results/eval_sft_vs_base.json


In [ ]:
import json, pandas as pd
d = json.load(open('results/eval_sft_vs_base.json'))
rows = [{'ckpt': c['name'],
         'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'cells': round(c['mean_cells'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')


## GRPO training — start from SFT adapter (π_1), curriculum layouts, shaped reward


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m training.train_grpo \
  --init-adapter ./ckpts/sft \
  --curriculum \
  --output-dir ./ckpts/grpo \
  --group-size 4 \
  --max-steps 50 \
  --save-steps 25 \
  --max-prompt-length 3500 \
  --max-completion-length 512


## Eval π_0 (base), π_1 (SFT), π_final (GRPO)


In [ ]:
!python -m training.evaluate \
  --checkpoints policy_0=BASE policy_1=./ckpts/sft policy_final=./ckpts/grpo \
  --samples-per-layout 4 --n-val 20 \
  --out results/eval_grpo_final.json

import json, pandas as pd
d = json.load(open('results/eval_grpo_final.json'))
rows = [{'ckpt': c['name'],
         'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'cells': round(c['mean_cells'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')
